In [ ]:
import pandas as pd
import re

In [ ]:
# Main execution
ref_df = pd.read_excel(r'..\acts_and_miracles\acts_and_miracles_prep.xlsx', sheet_name='acts_and_miracles', usecols="A,D")
scripture_df = pd.read_csv(r'..\scripture\asv.csv')

In [ ]:
ref_df

In [ ]:
scripture_df

In [ ]:
import pandas as pd
import re

def expand_reference(ref_str):
    """
    Parse and expand scripture references into individual verses.
    
    Handles formats like:
    - Genesis 1:3 (single verse)
    - Genesis 1:6-8 (verse range in same chapter)
    - Genesis 1:1-2:3 (cross-chapter range)
    - Genesis 1:24-25, 2:19 (multiple references)
    - Genesis 1:24, 26, 34, 2:19 (mixed: ch1v24, ch1v26, ch1v34, ch2v19)
    """
    results = []
    parts = ref_str.split(',')
    
    current_book = None
    current_chapter = None
    
    for part in parts:
        part = part.strip()
        
        # Check if this is a full reference (Book Chapter:Verse)
        full_match = re.match(r'([A-Za-z0-9\s]+)\s+(\d+):(\d+)(?:-(?:(\d+):)?(\d+))?', part)
        
        if full_match:
            # Full reference with book name
            current_book = full_match.group(1).strip()
            current_chapter = int(full_match.group(2))
            start_v = int(full_match.group(3))
            
            if full_match.group(5):  # Range exists
                if full_match.group(4):  # Cross-chapter range (Ch:V-Ch:V)
                    end_ch = int(full_match.group(4))
                    end_v = int(full_match.group(5))
                    # Generate all verses in range
                    for ch in range(current_chapter, end_ch + 1):
                        v_start = start_v if ch == current_chapter else 1
                        v_end = end_v if ch == end_ch else 999  # Max verse assumption
                        for v in range(v_start, v_end + 1):
                            results.append((current_book, ch, v))
                else:  # Same chapter range (Ch:V-V)
                    end_v = int(full_match.group(5))
                    for v in range(start_v, end_v + 1):
                        results.append((current_book, current_chapter, v))
            else:  # Single verse
                results.append((current_book, current_chapter, start_v))
        
        else:
            # Check for chapter:verse without book name (uses current book)
            ch_v_match = re.match(r'^(\d+):(\d+)(?:-(\d+))?$', part)
            if ch_v_match and current_book:
                current_chapter = int(ch_v_match.group(1))
                start_v = int(ch_v_match.group(2))
                
                if ch_v_match.group(3):  # Range in same chapter
                    end_v = int(ch_v_match.group(3))
                    for v in range(start_v, end_v + 1):
                        results.append((current_book, current_chapter, v))
                else:  # Single verse
                    results.append((current_book, current_chapter, start_v))
            
            # Check for just verse or verse range (uses current book and chapter)
            elif re.match(r'^\d+(?:-\d+)?$', part) and current_book and current_chapter:
                verse_match = re.match(r'^(\d+)(?:-(\d+))?$', part)
                if verse_match:
                    start_v = int(verse_match.group(1))
                    if verse_match.group(2):  # Verse range
                        end_v = int(verse_match.group(2))
                        for v in range(start_v, end_v + 1):
                            results.append((current_book, current_chapter, v))
                    else:  # Single verse
                        results.append((current_book, current_chapter, start_v))
    
    return results

def main():
    # Read reference data from Excel worksheet, extract columns A (Event ID) and D (Event Reference)
    ref_df = pd.read_excel(r'..\acts_and_miracles\acts_and_miracles_prep.xlsx', sheet_name='acts_and_miracles', usecols="A,D")
    
    # Read scripture CSV
    scripture_df = pd.read_csv(r'..\scripture\asv.csv')
    
    # Create lookup dictionary for scripture text
    scripture_dict = {
        (row['Book Name'], row['Chapter'], row['Verse']): row['Text']
        for _, row in scripture_df.iterrows()
    }
    
    # Process references and build result
    rows = []
    missing_count = 0
    
    for _, ref_row in ref_df.iterrows():
        event_id = ref_row['Event ID']
        reference = ref_row['Event Reference']
        
        if pd.isna(reference):
            continue
            
        # Expand reference to individual verses
        verses = expand_reference(reference)
        
        # Look up text for each verse
        for book, chapter, verse in verses:
            key = (book, chapter, verse)
            if key in scripture_dict:
                rows.append({
                    'Event ID': event_id,
                    'Event Reference': reference,
                    'Reference Code': f"{book} {chapter}:{verse}",
                    'Reference Text': scripture_dict[key]
                })
            else:
                missing_count += 1
    
    # Create result dataframe and save
    result_df = pd.DataFrame(rows)
    result_df.to_csv('expanded_references.csv', index=False)
    
    print(f"Created {len(result_df)} rows from {ref_df['Event ID'].nunique()} events")
    if missing_count > 0:
        print(f"Note: {missing_count} verse references could not be found in scripture file")
    print("\nFirst 5 rows:")
    print(result_df.head())
    
    # Example of handling complex references
    print("\nExample parsing:")
    test_ref = "Genesis 1:24, 26, 34, 2:19"
    print(f"'{test_ref}' expands to: {expand_reference(test_ref)}")
    
    return result_df

def create_concatenated_references(input_csv='expanded_references.csv', output_csv='concatenated_references.csv', input_df=None):
    """
    Create a new table where each reference has all its verses concatenated into a single text field.
    Format: [Book Name] [ch 1] [verse 1] text [verse 2] text [ch 2] [verse 1] text...
    
    Parameters:
    - input_csv: Path to the expanded references CSV (used if input_df is None)
    - output_csv: Path for the output concatenated CSV
    - input_df: Optional DataFrame to use instead of reading from CSV
    """
    # Use provided dataframe or read from CSV
    if input_df is not None:
        df = input_df
    else:
        df = pd.read_csv(input_csv)
    
    # Group by Event ID and Reference
    grouped = df.groupby(['Event ID', 'Event Reference'])
    
    result_rows = []
    
    for (event_id, reference), group in grouped:
        # Sort by Reference Code to ensure proper verse order
        group = group.sort_values('Reference Code')
        
        # Build concatenated text with formatting
        concatenated_text = ""
        current_book = None
        current_chapter = None
        
        for _, row in group.iterrows():
            ref_code = row['Reference Code']
            text = row['Reference Text']
            
            # Parse the reference code to get book, chapter, verse
            match = re.match(r'([A-Za-z0-9\s]+)\s+(\d+):(\d+)', ref_code)
            if match:
                book = match.group(1).strip()
                chapter = int(match.group(2))
                verse = int(match.group(3))
                
                # Add book marker if it's a new book
                if book != current_book:
                    concatenated_text += f"[{book}] "
                    current_book = book
                    current_chapter = None  # Reset chapter when book changes
                
                # Add chapter marker if it's a new chapter
                if chapter != current_chapter:
                    concatenated_text += f"[ch {chapter}] "
                    current_chapter = chapter
                
                # Add verse marker and text
                concatenated_text += f"[verse {verse}] {text} "
        
        result_rows.append({
            'Event ID': event_id,
            'Event Reference': reference,
            'Concatenated_Text': concatenated_text.strip()
        })
    
    # Create result dataframe and save
    result_df = pd.DataFrame(result_rows)
    #result_df.to_csv(output_csv, index=False)
    
    print(f"\nCreated concatenated references with {len(result_df)} rows")
    print(f"Saved to: {output_csv}")
    print("\nFirst example:")
    if len(result_df) > 0:
        first_row = result_df.iloc[0]
        print(f"Event ID: {first_row['Event ID']}")
        print(f"Reference: {first_row['Event Reference']}")
        print(f"Text preview (first 500 chars): {first_row['Concatenated_Text'][:500]}...")
    
    return result_df

def create_concatenated_references_alt(input_df=None, input_csv='expanded_references.csv', 
                                      output_csv='concatenated_references_alt.csv', 
                                      separator=' | ', include_markers=True):
    """
    Alternative version with customizable formatting.
    
    Parameters:
    - input_df: Optional DataFrame to use instead of reading from CSV
    - input_csv: Path to the expanded references CSV (used if input_df is None)
    - output_csv: Path for the output concatenated CSV
    - separator: String to separate verses (default: ' | ')
    - include_markers: Whether to include [book], [ch], [verse] markers
    """
    # Use provided dataframe or read from CSV
    df = input_df if input_df is not None else pd.read_csv(input_csv)
    
    # Group by Event ID and Reference
    grouped = df.groupby(['Event ID', 'Event Reference'])
    
    result_rows = []
    
    for (event_id, reference), group in grouped:
        # Sort to ensure proper order
        group = group.sort_values('Reference Code')
        
        if include_markers:
            # Build with markers as before
            concatenated_text = ""
            current_book = None
            current_chapter = None
            
            for _, row in group.iterrows():
                ref_code = row['Reference Code']
                text = row['Reference Text']
                
                match = re.match(r'([A-Za-z0-9\s]+)\s+(\d+):(\d+)', ref_code)
                if match:
                    book = match.group(1).strip()
                    chapter = int(match.group(2))
                    verse = int(match.group(3))
                    
                    if book != current_book:
                        concatenated_text += f"[{book}] "
                        current_book = book
                        current_chapter = None
                    
                    if chapter != current_chapter:
                        concatenated_text += f"[ch {chapter}] "
                        current_chapter = chapter
                    
                    concatenated_text += f"[v{verse}] {text}{separator}"
        else:
            # Simple concatenation without markers
            texts = []
            for _, row in group.iterrows():
                ref_code = row['Reference Code']
                text = row['Reference Text']
                texts.append(f"{ref_code}: {text}")
            concatenated_text = separator.join(texts)
        
        result_rows.append({
            'Event ID': event_id,
            'Event Reference': reference,
            'Concatenated_Text': concatenated_text.rstrip(separator).strip()
        })
    
    result_df = pd.DataFrame(result_rows)
    #result_df.to_csv(output_csv, index=False)
    
    print(f"\nCreated alternative concatenated references with {len(result_df)} rows")
    print(f"Saved to: {output_csv}")
    
    return result_df

if __name__ == "__main__":
    main()

In [ ]:
expanded_df = main()
expanded_df

In [ ]:
#concatenated_df = create_concatenated_references(input_df=expanded_df)
#concatenated_df

In [ ]:
# Added to correct columns for ERD design
erd_df = expanded_df[['Event ID', 'Reference Code']]
erd_df

In [ ]:
erd_df.to_csv(r'..\acts_and_miracles\DIM_events_to_verses.csv', index_label='EventVerseID')